## Silver Red Flags

normalized form: one row per siren, fiscal_year, flag_name where a red flag is TRUE

In [0]:
CATALOG = "credit_risk_project"
SILVER_SCHEMA = f"{CATALOG}.credit_risk_project_silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

In [0]:
from pyspark.sql import functions as F
from functools import reduce

df_bilans = spark.table(f"{SILVER_SCHEMA}.silver_bilans")

flag_columns = [
    ("flag_capitaux_propres_negatifs", "Capitalux propres ngeatifs"),
    ("flag_resultat_net_negatif", "Resultat net negatif"),
    ("flag_total_dettes_superieur_actif", "Total dettes > total actif"),
    ("flag_capital_insuffisant", "Capitaux propres < 50% du capital social"),
]

flag_frames = []
for col_name, description_txt in flag_columns:
    flag_frames.append(
        df_bilans
        .filter(F.col(col_name) == True)
        .select(
            "siren",
            "fiscal_year",
            "date_cloture",
            F.lit(col_name).alias("flag_name"),
            F.lit(description_txt).alias("flag_description"),
        )
    )

df_red_flags = (
    reduce(lambda a,b: a.unionByName(b), flag_frames)
    .withColumn("silver_ingested_at", F.current_timestamp())
)

print(f"silver_red_flags row count: {df_red_flags.count()}")
df_red_flags.display()


In [0]:
(
    df_red_flags.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{SILVER_SCHEMA}.silver_red_flags")
)

print(f"Saved {SILVER_SCHEMA}.silver_red_flags")

## Sanity Check


In [0]:
print("red_flags:", spark.table(f"{SILVER_SCHEMA}.silver_red_flags").count())

display(
    spark.sql(f"""
              SELECT c.company_name, r.fiscal_year, r.flag_name, r.flag_description
              FROM {SILVER_SCHEMA}.silver_red_flags r 
              JOIN {SILVER_SCHEMA}.silver_company c ON r.siren = c.siren
              ORDER BY c.company_name, r.fiscal_year, r.flag_name
              """)
)